<a href="https://colab.research.google.com/github/sharvi07/fake-job-detector/blob/main/notebooks/02_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, re
BASE = "/content/drive/MyDrive/fake-job-detector"
df = pd.read_csv(f"{BASE}/data/fake_job_postings.csv")

# 1. Drop exact duplicates (ignoring job_id)
df = df.drop_duplicates(subset=df.columns.drop("job_id")).reset_index(drop=True)

# 2. Merge text columns (nulls become empty strings)
text_cols = ["title", "company_profile", "description", "requirements", "benefits"]
df["text_raw"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# 3. Clean
def clean(t):
    t = re.sub(r"#(URL|EMAIL|PHONE)_[0-9a-f]+#", lambda m: f" {m.group(1).lower()}token ", t)
    t = re.sub(r"<[^>]+>", " ", t)               # HTML tags -> space
    t = re.sub(r"([a-z])([A-Z])", r"\1 \2", t)   # split glued words like overviewAker
    t = re.sub(r"[^a-zA-Z ]", " ", t).lower()
    return re.sub(r"\s+", " ", t).strip()

df["text"] = df["text_raw"].apply(clean)

print(df.shape)
print(df["fraudulent"].value_counts())
print(df["text"].iloc[0][:300])

Mounted at /content/drive
(17599, 20)
fraudulent
0    16743
1      856
Name: count, dtype: int64
marketing intern we re food and we ve created a groundbreaking and award winning cooking site we support connect and celebrate home cooks and give them everything they need in one place we have a top editorial business and engineering team we re focused on using technology to find new and better way


In [2]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
train, temp = train_test_split(df, test_size=0.30, stratify=df["fraudulent"], random_state=42)
# split temp into 15% val, 15% test
val, test = train_test_split(temp, test_size=0.50, stratify=temp["fraudulent"], random_state=42)

for name, d in [("train", train), ("val", val), ("test", test)]:
    print(name, d.shape, "fake %:", round(d["fraudulent"].mean() * 100, 2))

# Save everything (keep raw columns too, Person 2 needs them for rule-based features)
df.to_csv(f"{BASE}/data/cleaned_jobs.csv", index=False)
train.to_csv(f"{BASE}/data/train.csv", index=False)
val.to_csv(f"{BASE}/data/val.csv", index=False)
test.to_csv(f"{BASE}/data/test.csv", index=False)
print("saved")

train (12319, 20) fake %: 4.86
val (2640, 20) fake %: 4.89
test (2640, 20) fake %: 4.85
saved


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_union
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

# Word + character n-grams
vec = make_union(
    TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000, sublinear_tf=True),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_features=50000, sublinear_tf=True),
)

X_train = vec.fit_transform(train["text"])   # fit on train only
X_val = vec.transform(val["text"])
y_train, y_val = train["fraudulent"], val["fraudulent"]

clf = LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0)
clf.fit(X_train, y_train)

pred = clf.predict(X_val)
prob = clf.predict_proba(X_val)[:, 1]

print("Precision:", round(precision_score(y_val, pred), 3))
print("Recall:   ", round(recall_score(y_val, pred), 3))
print("F1:       ", round(f1_score(y_val, pred), 3))
print("PR-AUC:   ", round(average_precision_score(y_val, prob), 3))
print(confusion_matrix(y_val, pred))

Precision: 0.831
Recall:    0.876
F1:        0.853
PR-AUC:    0.924
[[2488   23]
 [  16  113]]


In [4]:
word_vec = vec.transformer_list[0][1]          # the word TF-IDF part
n_word = len(word_vec.get_feature_names_out())
names = word_vec.get_feature_names_out()
coefs = clf.coef_[0][:n_word]                  # word features come first

top = np.argsort(coefs)
print("PUSH TOWARDS REAL:", list(names[top[:15]]))
print("PUSH TOWARDS FAKE:", list(names[top[-15:]][::-1]))

PUSH TOWARDS REAL: ['our', 'companies', 'web', 'english', 'our clients', 'growing', 'it', 'recruitment', 'an', 'digital', 'of our', 'fun', 'team of', 'clients', 'new']
PUSH TOWARDS FAKE: ['link urltoken', 'below link', 'using below', 'apply using', 'link', 'data entry', 'line requirements', 'assistant', 'earn', 'northwestern', 'accion', 'administrative assistant', 'administrative', 'subsea', 'receptionist']


In [5]:
import joblib

X_test = vec.transform(test["text"])
y_test = test["fraudulent"]
pred_t = clf.predict(X_test)
prob_t = clf.predict_proba(X_test)[:, 1]

results = pd.DataFrame([{
    "model": "TF-IDF + LogReg (balanced)",
    "val_precision": round(precision_score(y_val, pred), 3),
    "val_recall": round(recall_score(y_val, pred), 3),
    "val_f1": round(f1_score(y_val, pred), 3),
    "test_precision": round(precision_score(y_test, pred_t), 3),
    "test_recall": round(recall_score(y_test, pred_t), 3),
    "test_f1": round(f1_score(y_test, pred_t), 3),
    "test_pr_auc": round(average_precision_score(y_test, prob_t), 3),
}])
print(results.T)
print(confusion_matrix(y_test, pred_t))

results.to_csv(f"{BASE}/baseline_results.csv", index=False)
joblib.dump((vec, clf), f"{BASE}/models/baseline_tfidf_lr.pkl")
print("saved")

                                         0
model           TF-IDF + LogReg (balanced)
val_precision                        0.831
val_recall                           0.876
val_f1                               0.853
test_precision                       0.748
test_recall                          0.859
test_f1                                0.8
test_pr_auc                          0.904
[[2475   37]
 [  18  110]]
saved
